# Two-stage Unstain → virtual H&E: held-out WSI cohort test

학습과 동일한 slide-wise split을 재현하여 **8개 held-out WSI 전체**를 `2.0 MPP`에서 평가합니다. Stage 1과 Stage 2 모두 `latest.pt`를 사용합니다. 각 WSI의 원본 H&E를 저장된 affine matrix로 Unstain 좌표계에 정합하고, 타일 → 슬라이드 → 독립 case 순으로 집계합니다.

전체 83개 WSI가 아니라 held-out 8개만 기본 test cohort로 사용하는 이유는 학습 WSI가 test 결과에 섞이는 data leakage를 방지하기 위해서입니다.

In [ ]:
from pathlib import Path

import torch
from IPython.display import Image as NotebookImage, display

from two_stage_inference import load_two_stage_models
from two_stage_wsi_batch_evaluation import run_heldout_wsi_evaluation
from wsi_publication_evaluation import PRIMARY_METRICS, discover_heldout_wsi

torch.set_grad_enabled(False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
DATA_DIR = Path('../../data/HnE_n_UNStaining/patch_dataset_mpp05_2048')
UNSTAIN_WSI_DIR = Path('../../data/HnE_n_UNStaining/Un-Stained-tiff')
HNE_WSI_DIR = Path('../../data/HnE_n_UNStaining/C_Stained-tiff')
REGISTRATION_DIR = Path('../../data/HnE_n_UNStaining/registration_results/matrices')

# best.pt가 아니라 latest.pt를 명시적으로 사용합니다.
STRUCTURE_CKPT = Path('../../model/Unstain2HnE_two_stage_v2/structure/latest.pt')
COLOR_CKPT = Path('../../model/Unstain2HnE_two_stage_v4/color/latest.pt')
OUTPUT_DIR = Path('../../results/Unstain2HnE_two_stage_v4/wsi_test_all_latest')
REUSE_EXISTING = True

structure, colorizer, metadata = load_two_stage_models(
    STRUCTURE_CKPT, COLOR_CKPT, device
)
assert metadata['target_mpp'] == 2.0, metadata
manifest = discover_heldout_wsi(
    DATA_DIR, UNSTAIN_WSI_DIR, HNE_WSI_DIR, REGISTRATION_DIR,
    image_ext='png', val_fraction=0.1, seed=42,
)
print(f"Structure latest epoch: {metadata['structure_epoch']}")
print(f"Color latest epoch: {metadata['color_epoch']}")
print(f"Held-out cohort: {manifest['case_id'].nunique()} cases / {len(manifest)} slides")
display(manifest)
assert manifest['ready'].all(), manifest.loc[~manifest['ready']]

## Metrics and statistical unit

- `ssim_full_rgb`: 논문 비교용 표준 full-tile RGB SSIM
- `ssim_tissue_rgb`: 흰 배경을 제외한 엄격한 조직 전용 SSIM
- `ssim_coarse_rgb`: 4배 area downsampling 후 거시 구조 SSIM
- `psnr_full_db`, `psnr_tissue_db`: full/tissue PSNR 분리
- `LPIPS`: AlexNet perceptual distance
- `FID/KID`: 모든 held-out WSI 조직 타일의 Inception feature distribution 비교
- `ΔE2000`, H/E concentration MAE: 염색 및 색상 충실도
- `gradient correlation`, Laplacian energy error: 경계와 선명도
- `h_spatial_corr`: 2.0 MPP에서 hematoxylin 공간 밀도 일치도
- `tissue Dice`: 조직 범위 보존

신뢰구간과 paired Wilcoxon 검정은 타일을 독립 표본으로 취급하지 않습니다. 먼저 슬라이드 평균을 만들고, 동일 환자/case의 복수 슬라이드를 다시 평균한 뒤 **case-level bootstrap 95% CI**와 검정을 계산합니다.

In [ ]:
results = run_heldout_wsi_evaluation(
    manifest, structure, colorizer, metadata, OUTPUT_DIR,
    reuse_existing=REUSE_EXISTING,
    inference_batch_size=4, deep_batch_size=8,
    tile_size=512, stride=512, minimum_tissue_fraction=0.10,
    bootstrap_iterations=5000,
)
display(results['run_manifest'])

## Cohort-level publication report

`full RGB SSIM/FID/KID`는 ViT-Stain 같은 기존 문헌과 형식상 비교하기 위한 값입니다. 그러나 다른 MPP, FOV, split, registration 및 preprocessing에서 나온 문헌 수치와 직접적인 우열을 주장하면 안 됩니다. `tissue-only SSIM`, hematoxylin spatial correlation과 색상/경계 지표를 함께 보고하여 흰 배경에 의한 점수 부풀림과 hallucinated cellular texture를 드러냅니다.

현재 모델 출력이 2.0 MPP이므로 개별 핵 count/instance segmentation은 정식 지표로 사용하지 않습니다. 해당 평가는 0.5 MPP 모델 또는 실제 고해상도 출력으로 별도 수행해야 합니다.

In [ ]:
primary_summary = results['cohort_summary'].query('metric in @PRIMARY_METRICS')
primary_improvement = results['cohort_improvement'].query('metric in @PRIMARY_METRICS')
display(primary_summary.round(4))
display(primary_improvement.round(4))
display(results['distribution_metrics'].round(4))
display(NotebookImage(filename=str(results['outputs']['cohort_metric_figure'])))
display(NotebookImage(filename=str(results['outputs']['cohort_overview'])))
print('Saved outputs:')
for name, path in results['outputs'].items():
    print(f'{name}: {path}')